In [1]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
from urllib.parse import quote_plus
import pandas as pd
import os
import re 



In [2]:
load_dotenv()

# quote_plus special characters ko encode karta hai
password = quote_plus(os.getenv('DB_PASSWORD'))

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{password}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)

df = pd.read_sql("SELECT * FROM raw_jobs", engine)
print(df.shape)
print(f"Loaded {len(df)} clean rows")
print(f"\nNull values:\n{df.isnull().sum()}")
print(f"\nData types:\n{df.dtypes}")

(391, 11)
Loaded 391 clean rows

Null values:
id                      0
job_title               0
company_name            0
job_description        10
job_employment_type     0
job_city               77
job_country             1
job_apply_link          0
posted_date             0
created_at              0
role_searched           0
dtype: int64

Data types:
id                              int64
job_title                      object
company_name                   object
job_description                object
job_employment_type            object
job_city                       object
job_country                    object
job_apply_link                 object
posted_date            datetime64[ns]
created_at             datetime64[ns]
role_searched                  object
dtype: object


In [3]:
df.shape

(391, 11)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391 entries, 0 to 390
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id                   391 non-null    int64         
 1   job_title            391 non-null    object        
 2   company_name         391 non-null    object        
 3   job_description      381 non-null    object        
 4   job_employment_type  391 non-null    object        
 5   job_city             314 non-null    object        
 6   job_country          390 non-null    object        
 7   job_apply_link       391 non-null    object        
 8   posted_date          391 non-null    datetime64[ns]
 9   created_at           391 non-null    datetime64[ns]
 10  role_searched        391 non-null    object        
dtypes: datetime64[ns](2), int64(1), object(8)
memory usage: 33.7+ KB


In [5]:
df.describe()

,id,posted_date,created_at
count,391.000000,391,391
mean,196.000000,2026-02-26 10:59:13.964194560,2026-03-01 15:15:24.562852352
min,1.000000,2026-02-22 00:00:00,2026-02-28 16:33:14.406315
25%,98.500000,2026-02-24 00:00:00,2026-02-28 16:33:14.406315008
50%,196.000000,2026-02-26 00:00:00,2026-02-28 16:41:23.856080896
75%,293.500000,2026-02-27 00:00:00,2026-02-28 16:41:23.856080896
max,391.000000,2026-04-01 10:00:00,2026-04-02 00:09:09.186136
std,113.016223,NaN,NaN


In [6]:
df.isnull().sum()

id                      0
job_title               0
company_name            0
job_description        10
job_employment_type     0
job_city               77
job_country             1
job_apply_link          0
posted_date             0
created_at              0
role_searched           0
dtype: int64

In [7]:
df.duplicated().sum()


np.int64(0)

In [9]:
df.dtypes

id                              int64
job_title                      object
company_name                   object
job_description                object
job_employment_type            object
job_city                       object
job_country                    object
job_apply_link                 object
posted_date            datetime64[ns]
created_at             datetime64[ns]
role_searched                  object
dtype: object

In [8]:
df.isnull()

,id,job_title,company_name,job_description,job_employment_type,job_city,job_country,job_apply_link,posted_date,created_at,role_searched
0,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...
386,False,False,False,True,False,False,False,False,False,False,False
387,False,False,False,True,False,True,False,False,False,False,False
388,False,False,False,True,False,False,False,False,False,False,False
389,False,False,False,True,False,False,False,False,False,False,False


# ─── EXPLORATION SUMMARY ───────────────────────
# Shape: 381 rows, 11 columns
# Nulls: job_city (75), job_country (1)
# Duplicates: 0
# Dtypes: all object — needs conversion
# Issue: no salary column from free API tier
# ───────────────────────────────────────────────

In [9]:
print(df[["job_city"]].value_counts().head(3))
print(df["role_searched"].value_counts().head(3))

job_city 
Bengaluru    70
Gurugram     42
Chennai      39
Name: count, dtype: int64
role_searched
Data Analyst      117
Data Scientist     88
Data Engineer      66
Name: count, dtype: int64


In [10]:
df['job_employment_type']=df['job_employment_type'].str.strip().str.title()
print(df["job_employment_type"].value_counts())

job_employment_type
Full-Time     390
Contractor      1
Name: count, dtype: int64


In [11]:
df["job_employment_type"].head(3)

0    Full-Time
1    Full-Time
2    Full-Time
Name: job_employment_type, dtype: object

In [12]:
df.duplicated().sum()

np.int64(0)

In [13]:
# job_title + employer_name combination duplicates
df.duplicated(subset=["job_title", "job_city", "company_name"]).sum()


np.int64(317)

In [14]:
df.drop_duplicates(subset=["job_title", "job_city", "company_name"], keep='first', inplace=True) 
print(df.shape)

(74, 11)


In [15]:
print(f'final shape: {df.shape}')
print(f'final null values:\n{df.isnull().sum()}')
print(f'final data types:\n{df.dtypes}')


final shape: (74, 11)
final null values:
id                      0
job_title               0
company_name            0
job_description        10
job_employment_type     0
job_city               17
job_country             1
job_apply_link          0
posted_date             0
created_at              0
role_searched           0
dtype: int64
final data types:
id                              int64
job_title                      object
company_name                   object
job_description                object
job_employment_type            object
job_city                       object
job_country                    object
job_apply_link                 object
posted_date            datetime64[ns]
created_at             datetime64[ns]
role_searched                  object
dtype: object


In [16]:
df = pd.read_sql("SELECT * FROM raw_jobs", engine)

In [17]:
df["job_city"].fillna("Unknown", inplace=True)

C:\Users\aadit\AppData\Local\Temp\ipykernel_8404\4101281386.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["job_city"].fillna("Unknown", inplace=True)


In [20]:
df["posted_date"]=pd.to_datetime(df["posted_date"], errors='coerce')
print(df.isnull().sum())
print(df['posted_date'].head(3))

id                     0
job_title              0
company_name           0
job_description        0
job_employment_type    0
job_city               0
job_country            0
job_apply_link         0
posted_date            0
created_at             0
role_searched          0
dtype: int64
0   2026-02-27 00:00:00
1   2026-02-27 00:00:00
2   2026-02-28 03:00:00
Name: posted_date, dtype: datetime64[ns]


In [18]:
df["job_city"]=df["job_city"].str.strip().str.title()
df["job_country"]=df["job_country"].str.strip().str.title()
df["job_title"]=df["job_title"].str.strip().str.title() 
df["role_searched"]=df["role_searched"].str.strip().str.title()
df["job_description"]=df["job_description"].str.strip()

In [19]:
df["job_country"].fillna("India", inplace=True)

C:\Users\aadit\AppData\Local\Temp\ipykernel_8404\2978140326.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["job_country"].fillna("India", inplace=True)


In [20]:
df.columns

Index(['id', 'job_title', 'company_name', 'job_description',
       'job_employment_type', 'job_city', 'job_country', 'job_apply_link',
       'posted_date', 'created_at', 'role_searched'],
      dtype='object')

In [21]:
df.rename(columns={'employer_name': 'company_name'}, inplace=True)
print(df.columns.tolist())

['id', 'job_title', 'company_name', 'job_description', 'job_employment_type', 'job_city', 'job_country', 'job_apply_link', 'posted_date', 'created_at', 'role_searched']


In [22]:
df.duplicated(subset='job_apply_link').sum()

np.int64(327)

In [26]:
df['job_apply_link'].head(70)

0     https://in.linkedin.com/jobs/view/senior-data-...
1     https://jobs.comcast.com/job/chennai/data-anal...
2     https://www.efinancialcareers.com/jobs-India-B...
3     https://careers.mastercard.com/us/en/job/R-262...
4     https://www.shine.com/jobs/data-analyst-data-s...
                            ...                        
59    https://www.glassdoor.co.in/job-listing/data-a...
60    https://www.monster.com/job-openings/databrick...
61    https://jobright.ai/jobs/info/699cb5b481476f61...
62    https://bristol.us.expertini.com/job/intl-indi...
63    https://tallo.com/jobs/technology/data-enginee...
Name: job_apply_link, Length: 64, dtype: object

In [23]:
df.drop_duplicates(subset=["job_apply_link"], keep='first', inplace=True)

In [24]:
df.duplicated(['job_apply_link']).sum()

np.int64(0)

In [25]:
print(f"Final shape: {df.shape}")
print(f"Null values:\n{df.isnull().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")

Final shape: (74, 11)
Null values:
id                      0
job_title               0
company_name            0
job_description        10
job_employment_type     0
job_city                0
job_country             0
job_apply_link          0
posted_date             0
created_at              0
role_searched           0
dtype: int64
Duplicates: 0


In [30]:
print(f"\nSample:\n{df.head(3)}")


Sample:
   id                                      job_title  \
0   1    Senior Data Analyst (Qa & Automation Focus)   
1   2                                 Data Analyst 3   
2   3  Senior Financial Data Analyst | Bangalore, In   

                   company_name  \
0  SID Information Technologies   
1                       Comcast   
2                       Moody's   

                                     job_description job_employment_type  \
0  Role: Senior Data Analyst (QA & Automation Foc...           Full-Time   
1  Comcast brings together the best in media and ...           Full-Time   
2  At Moody's, we unite the brightest minds to tu...           Full-Time   

    job_city job_country                                     job_apply_link  \
0  Hyderabad          In  https://in.linkedin.com/jobs/view/senior-data-...   
1    Chennai          In  https://jobs.comcast.com/job/chennai/data-anal...   
2  Bengaluru          In  https://www.efinancialcareers.com/jobs-India-B...   

   

In [26]:
df.to_csv("cleaned_jobs.csv", index=False)
print("Cleaned data saved to cleaned_jobs.csv")

Cleaned data saved to cleaned_jobs.csv


In [27]:
df.to_sql(
    'transformed_jobs',  
    engine,              
    if_exists='replace', 
    index=False          
)
print(f"✅ {len(df)} clean jobs loaded into transformed_jobs!")

✅ 74 clean jobs loaded into transformed_jobs!


In [28]:
skills_list = [
    # Programming 
    'python', 'sql', 'java', 'scala',
    # DE Tools
    'airflow', 'spark', 'kafka', 'docker',
    'kubernetes', 'dbt',
    # Databases
    'postgresql', 'mysql', 'mongodb',
    'cassandra', 'redis', 'snowflake',
    # Cloud
    'aws', 'azure', 'gcp', 'redshift',
    'bigquery', 'databricks',
    # DA Tools
    'pandas', 'numpy', 'matplotlib',
    'seaborn', 'tableau', 'power bi',
    # ML
    'machine learning', 'deep learning',
    'tensorflow', 'pytorch', 'scikit-learn',
    # Others
    'git', 'linux', 'rest api', 'etl',
    'data pipeline', 'excel'
]

In [29]:
def extract_skills(description):
    if not description:
        return []
    
    description = description.lower()  # lowercase for matching
    found_skills = []
    
    for skill in skills_list:
        # Minimum 2 characters
        if len(skill) < 2:
            continue
        pattern = r'\b' + re.escape(skill) + r'\b'
        if re.search(pattern, description):
            found_skills.append(skill)
    
    return found_skills

In [30]:
sample=df["job_description"].iloc[0]
print(extract_skills(sample))

['python', 'sql', 'aws', 'tableau', 'etl']


In [33]:
from collections import Counter

df['skills_found'] = df['job_description'].apply(extract_skills)

all_skills = []
for skills in df['skills_found']:
    all_skills.extend(skills)

skill_counts = Counter(all_skills)
print(f"Average skills per job: {df['skills_found'].apply(len).mean():.1f}")


Average skills per job: 4.5


In [34]:
job_skills_rows = []

for _, row in df.iterrows():
    for skill in row['skills_found']:
        job_skills_rows.append({
            'job_id': row['id'],
            'job_title': row['job_title'],
            'company_name': row['company_name'],
            'skill': skill,
            'role_searched': row['role_searched']
        })

job_skills_df = pd.DataFrame(job_skills_rows)
print(f"Total skill records: {len(job_skills_df)}")
print(job_skills_df.head(10))

Total skill records: 330
   job_id                                    job_title  \
0       1  Senior Data Analyst (Qa & Automation Focus)   
1       1  Senior Data Analyst (Qa & Automation Focus)   
2       1  Senior Data Analyst (Qa & Automation Focus)   
3       1  Senior Data Analyst (Qa & Automation Focus)   
4       1  Senior Data Analyst (Qa & Automation Focus)   
5       2                               Data Analyst 3   
6       2                               Data Analyst 3   
7       2                               Data Analyst 3   
8       2                               Data Analyst 3   
9       2                               Data Analyst 3   

                   company_name    skill   role_searched  
0  SID Information Technologies   python  Data Scientist  
1  SID Information Technologies      sql  Data Scientist  
2  SID Information Technologies      aws  Data Scientist  
3  SID Information Technologies  tableau  Data Scientist  
4  SID Information Technologies      etl 

In [35]:
job_skills_df.to_sql(
    'job_skills',
    engine,
    if_exists='replace',
    index=False
)
print(f"✅ {len(job_skills_df)} skill records loaded!")

✅ 330 skill records loaded!
